# Credit Risk EDA Walkthrough
This notebook reads outputs created by the reproducible Python pipeline. Run `python run_pipeline.py --mode csv-only` first.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
loans = pd.read_csv(ROOT / 'data/processed/loans_clean.csv', parse_dates=['origination_date'])
metrics = pd.read_csv(ROOT / 'data/outputs/model_metrics.csv')
deciles = pd.read_csv(ROOT / 'dashboard/data/risk_decile_lift.csv')
audit = json.loads((ROOT / 'data/outputs/data_audit_report.json').read_text())
loans.shape, audit['default_rate']

## Target balance and temporal shift
Use origination cohorts because later years have materially different default prevalence.

In [ ]:
display(loans['defaulted'].value_counts().rename_axis('defaulted').to_frame('rows'))
yearly = loans.groupby('origination_year')['defaulted'].agg(['count','sum','mean'])
display(yearly)
yearly['mean'].plot(marker='o', title='Eventual Default Rate by Origination Year')
plt.ylabel('Default rate')
plt.show()

## Rating and sector segmentation
These are predictive/descriptive relationships, not causal effects.

In [ ]:
rating_order = ['AAA','AA','A','BBB','BB','B','CCC']
rating_risk = loans.groupby('initial_rating')['defaulted'].mean().reindex(rating_order)
rating_risk.plot(kind='bar', color='#1F4E78', title='Observed Default Rate by Initial Rating')
plt.ylabel('Default rate')
plt.show()
display(loans.groupby('sector').agg(loans=('loan_id','nunique'), ead=('ead','sum'), default_rate=('defaulted','mean')).sort_values('default_rate', ascending=False))

## Model evaluation
PR-AUC is primary because defaults are imbalanced. The test period is 2022–2023.

In [ ]:
display(metrics)
sns.barplot(data=deciles, x='risk_decile', y='default_rate', color='#1F4E78')
plt.title('Test Default Rate by Portfolio Risk Decile')
plt.ylabel('Observed default rate')
plt.show()

## Leakage reminder
Never use `default_date`, `survival_months`, `recovery_rate`, or observed loss fields as model inputs. Existing annual PD/LGD/EL/RWA outputs are also excluded to avoid circularity.